# M2 · Matrices as Transformations — companion notebook

> **Play with this.** A *demonstration, not an assessment* — the module's real assessment is its problem set. This notebook drills the one skill the module says matters most in practice: **reading shapes and shape errors**. It deliberately breaks computations so you can practise diagnosing them, then builds up to the exact shape choreography of an attention computation.

Companion to the **Matrices as Transformations** module of the Mathematical Foundations track at [llmsforsocialscience.net](https://llmsforsocialscience.net/).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2)

## 1 · A matrix is a function

Watch a matrix act on a whole grid of points at once. The columns of $A$ are where the basis vectors land — check the printed columns against the arrows in the plot.

In [ ]:
A = np.array([[1.0, 0.8],
              [0.0, 1.0]])   # a shear — try a rotation or the collapse matrix [[1, .5], [2, 1]]

# a grid of points, as a 2 x N matrix (each column is a point)
g = np.linspace(-2, 2, 9)
pts = np.array([[x, y] for x in g for y in g]).T
out = A @ pts                     # the function, applied to every point at once

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), sharex=True, sharey=True)
axes[0].scatter(pts[0], pts[1], s=12); axes[0].set_title("before")
axes[1].scatter(out[0], out[1], s=12, color="tab:red"); axes[1].set_title("after A")
for ax in axes:
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_aspect("equal")
axes[1].quiver([0, 0], [0, 0], A[0], A[1], angles="xy", scale_units="xy", scale=1, color=["tab:blue", "tab:green"])
plt.tight_layout(); plt.show()

print("columns of A (= images of the basis vectors):")
print(A[:, 0], "and", A[:, 1])

## 2 · Shape reading, drilled

The module's rule: $(n \times d)(d \times k) \to (n \times k)$ — inner dimensions must match and annihilate. Predict each result below **before** running the cell.

In [ ]:
P = rng.standard_normal((3, 4))
Q = rng.standard_normal((4, 4))
R = rng.standard_normal((4, 2))
v = rng.standard_normal(4)

for name, expr in [("P @ Q", lambda: P @ Q), ("Q @ P", lambda: Q @ P),
                   ("P @ R", lambda: P @ R), ("R @ P", lambda: R @ P),
                   ("Q @ v", lambda: Q @ v), ("P.T @ P", lambda: P.T @ P),
                   ("P @ P.T", lambda: P @ P.T)]:
    try:
        print(f"{name:9s} -> shape {expr().shape}")
    except ValueError as e:
        print(f"{name:9s} -> ERROR: {e}")

Read the two errors carefully: numpy names the two mismatched dimensions. That message is a *diagnosis*, and learning to read it is the point of this notebook.

Note also `P.T @ P` (4×4) versus `P @ P.T` (3×3): **both run without error and give different objects.** Transposing until the error goes away — without knowing which product you meant — is the classic silent bug, worse than a crash because nothing warns you.

## 3 · Deliberately broken: fix each one

Three realistic computations, each with a shape bug. For each: run it, read the error, name the fix, then apply it. Solutions in the cell after — no peeking.

In [ ]:
# (a) Similarity of every document against a query
docs = rng.standard_normal((100, 64))    # 100 documents, 64-dim embeddings
query = rng.standard_normal((1, 64))     # one query
try:
    scores = docs @ query                 # BROKEN — read the error, then fix
except ValueError as e:
    print("(a)", e)

# (b) Batch of vectors through a linear layer
W = rng.standard_normal((32, 64))         # layer: 64 -> 32
batch = rng.standard_normal((16, 64))     # 16 inputs
try:
    h = W @ batch                          # BROKEN
except ValueError as e:
    print("(b)", e)

# (c) All pairwise similarities within a batch
X = rng.standard_normal((32, 768))
try:
    sims = X @ X                           # BROKEN
except ValueError as e:
    print("(c)", e)

In [ ]:
# Fixes — with the shape story spelled out
scores = docs @ query.T        # (100 x 64)(64 x 1) -> (100 x 1): one score per document
h = batch @ W.T                # (16 x 64)(64 x 32) -> (16 x 32): batch stays first
sims = X @ X.T                 # (32 x 768)(768 x 32) -> (32 x 32): every vector vs every other
print(scores.shape, h.shape, sims.shape)

## 4 · One-hot times a matrix is a lookup

The module's column view, live: a one-hot vector times an embedding matrix selects a row — and the same multiply with a *soft* weight vector returns a mixture, which is attention's output step.

In [ ]:
E = rng.standard_normal((5, 3)).round(2)   # vocabulary of 5 words, 3-dim embeddings
one_hot = np.array([0, 0, 1, 0, 0])
print("E:\n", E)
print("one_hot @ E:  ", one_hot @ E)
print("E[2] (lookup):", E[2])

soft = np.array([0.05, 0.05, 0.7, 0.1, 0.1])   # a soft distribution instead
print("soft @ E:     ", (soft @ E).round(3), "  <- a weighted mixture of embeddings")

## 5 · Composition, and order

Rotate-then-stretch versus stretch-then-rotate, on the same unit square. Matrix multiplication is composition, and composition of actions is not commutative.

In [ ]:
th = np.pi / 2
Rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
D = np.array([[2.0, 0.0], [0.0, 1.0]])

square = np.array([[0, 1, 1, 0, 0], [0, 0, 1, 1, 0]])
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True, sharey=True)
for ax, (M, title) in zip(axes, [(np.eye(2), "unit square"), (Rot @ D, "stretch, then rotate (RD)"), (D @ Rot, "rotate, then stretch (DR)")]):
    s = M @ square
    ax.plot(s[0], s[1]); ax.fill(s[0], s[1], alpha=0.2)
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

print("RD:\n", (Rot @ D).round(2))
print("DR:\n", (D @ Rot).round(2))

## 6 · The attention shape choreography

The full shape story of one attention head, with a batch dimension — every step is a shape you can now predict. This is the payoff of treating shapes as a type system: an unfamiliar computation becomes readable from its signatures alone.

In [ ]:
batch, seq, d, dk = 4, 10, 64, 16

X   = rng.standard_normal((batch, seq, d))    # hidden states: batch x seq x d
W_q = rng.standard_normal((d, dk))            # learned projections
W_k = rng.standard_normal((d, dk))
W_v = rng.standard_normal((d, dk))

Q = X @ W_q                                   # (b, s, d)(d, dk)   -> (b, s, dk)
K = X @ W_k
V = X @ W_v
scores = Q @ K.transpose(0, 2, 1) / np.sqrt(dk)  # (b, s, dk)(b, dk, s) -> (b, s, s)
weights = np.exp(scores) / np.exp(scores).sum(axis=-1, keepdims=True)  # softmax rows
out = weights @ V                             # (b, s, s)(b, s, dk) -> (b, s, dk)

for name, arr in [("X", X), ("Q", Q), ("scores", scores), ("weights", weights), ("out", out)]:
    print(f"{name:8s} {arr.shape}")

Read the middle line again: `scores` is (batch, seq, seq) — every position against every position, the row view at scale — and `out` is `weights @ V`, a soft mixture of value vectors: the column view with soft weights, exactly section 4's lookup generalised. One attention head is this module, twice.

---

**Next:** M3 · Projection and Subspaces — where OLS, which you already use, turns out to be geometry you now own.